# QDM Bias Correction — CanESM5 vs ERA5

Trains Quantile Delta Mapping (Cannon et al. 2015) on ERA5 1980–2010 vs CanESM5 historical,  
then applies it to SSP245 and SSP585 (2015–2100).

| Variable | Transform | Unit |
|----------|-----------|------|
| tas      | additive  | K    |
| tasmax   | additive  | K    |
| sfcWind  | log + additive | m s⁻¹ |
| rsds     | log + additive | W m⁻² |

In [2]:
import os, sys, glob
from pathlib import Path

ENV_DIR = Path(sys.prefix)
_mk = ENV_DIR / "Library" / "lib" / "esmf.mk"   # Windows conda layout
if not _mk.exists():
    _mk = ENV_DIR / "lib" / "esmf.mk"            # Linux / Mac
os.environ["ESMFMKFILE"] = str(_mk)
print(f"ESMFMKFILE = {os.environ['ESMFMKFILE']}  (exists: {_mk.exists()})")

import xesmf as xe

import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
from xclim import sdba
from xclim.sdba import processing as sdba_proc
from xarray.coding.calendar_ops import convert_calendar

# Set ESMFMKFILE before importing xesmf

ERA5_DIR   = Path("../data/raw/era5_daily")
CMIP_DIR   = Path("../data/raw/CanESM5")
OUTPUT_DIR = Path("../data/proc/cmip6_bc")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GCM   = "CanESM5"
RUN   = "r10i1p1f1"
SSPS  = ["ssp245", "ssp585"]
TRAIN = slice("1980-01-01", "2010-12-31")

ESMFMKFILE = c:\Users\colin\anaconda3\envs\xesmf_env\Library\lib\esmf.mk  (exists: True)


## 0. Smoke test — full QDM pipeline on one pixel (tas, 1980 training, ssp245 2015)

In [4]:
_LAT, _LON = 20.0, 80.0      # pixel to inspect after regridding — change freely
_TRAIN_YR  = "1980"
_FUT_YR    = "2015"

# ── ERA5: 1 year as Dataset, drop extra scalar coords that confuse xesmf ─────────
_era5_ds = (
    xr.open_dataset(ERA5_DIR / "era5_tas_1980_2020.nc")
    .sel(time=slice(f"{_TRAIN_YR}-01-01", f"{_TRAIN_YR}-12-31"))
    .sortby("lat").sortby("lon")
)
_era5_ds = _era5_ds.sel(lat=_LAT, lon=_LON, method="nearest")

_era5_ds = _era5_ds.drop_vars([c for c in _era5_ds.coords
                                if c not in {"time", "lat", "lon"}], errors="ignore")
_era5_ds = convert_calendar(_era5_ds, "noleap")
_era5_ds.attrs["units"] = "K"
_era5_ds = _era5_ds.dropna(dim="time", how="all")
_era5_ds



<xarray.Dataset> Size: 4kB
Dimensions:  (time: 365)
Coordinates:
    lat      float64 8B 20.0
    lon      float64 8B 80.0
  * time     (time) object 3kB 1980-01-01 00:00:00 ... 1980-12-31 00:00:00
Data variables:
    tas      (time) float32 1kB 296.2 296.7 296.8 295.2 ... 294.9 295.7 294.5
Attributes:
    units:    K

In [14]:
# ── CanESM5 historical: 1 year, used as training data and as target grid ──────────
_pattern = str(CMIP_DIR / f"tas_day_{GCM}_historical_{RUN}*_india.nc")
_files = sorted(glob.glob(_pattern))
if not _files:
    raise FileNotFoundError(_pattern)
_hist_ds = (
    xr.open_mfdataset(_files, combine="by_coords")[["tas"]]
    .sel(time=slice(f"{_TRAIN_YR}-01-01", f"{_TRAIN_YR}-12-31"))
    .sortby("lat").sortby("lon")
)
_hist_ds = _hist_ds.sel(lat=_LAT, lon=_LON, method="nearest")
_hist_ds["time"] = _hist_ds.indexes["time"].floor("D")
_hist_ds["tas"].attrs["units"] = "K"
print(f"CanESM5: {dict(_hist_ds['tas'].sizes)}"
      f"  lat {float(_hist_ds.lat.min()):.2f}–{float(_hist_ds.lat.max()):.2f}")


# ── Select one pixel on the shared grid ──────────────────────────────────────────
_e = _era5_ds["tas"]
_h = _hist_ds["tas"]
print(f"Pixel  lat={float(_e.lat):.2f} lon={float(_e.lon):.2f}")

#change lat and lon to have the same
_e = _e.assign_coords(lat=_h.lat.values, lon=_h.lon.values)



CanESM5: {'time': 365}  lat 20.93–20.93
Pixel  lat=20.00 lon=80.00


In [15]:

# ── Train QDM ────────────────────────────────────────────────────────────────────
_QM = sdba.QuantileDeltaMapping.train(_e, _h, nquantiles=5, kind="+", group="time.month")
print("QDM trained OK")

# ── Future: ssp245, 1 year, same pixel ───────────────────────────────────────────
_pattern_fut = str(CMIP_DIR / f"tas_day_{GCM}_ssp245_{RUN}*_india.nc")
_files_fut = sorted(glob.glob(_pattern_fut))
if not _files_fut:
    raise FileNotFoundError(_pattern_fut)
_f = (
    xr.open_mfdataset(_files_fut, combine="by_coords")["tas"]
    .sel(lat=_LAT, lon=_LON, method="nearest")
    .sel(time=slice(f"{_FUT_YR}-01-01", f"{_FUT_YR}-12-31"))
)
_f["time"] = _f.indexes["time"].floor("D")
_f.attrs["units"] = "K"
_f = _f.assign_coords(lat=_h.lat, lon=_h.lon)   # same override for future
_f = _f.load()
print(f"Future pixel n={len(_f.time)}")

# ── Apply QDM ────────────────────────────────────────────────────────────────────
_bc = _QM.adjust(_f, interp="linear", extrapolation="constant")

print(f"\nSmoke test PASSED")
print(f"  raw  tas: min={float(_f.min()):.2f}  mean={float(_f.mean()):.2f}  max={float(_f.max()):.2f} K")
print(f"  bc   tas: min={float(_bc.min()):.2f}  mean={float(_bc.mean()):.2f}  max={float(_bc.max()):.2f} K")



QDM trained OK
Future pixel n=365

Smoke test PASSED
  raw  tas: min=285.00  mean=301.77  max=313.75 K
  bc   tas: min=287.51  mean=301.05  max=312.72 K


NameError: name '_regridder' is not defined

## 1. Variable-by-variable QDM bias correction

Each variable is loaded, trained, and written to disk before the next one starts — keeping only one variable's data in RAM at a time.

In [ ]:
VAR_CFG = {
    # var       kind   log    jitter   units
    "tas":     ("+", False, None,   "K"),
    "tasmax":  ("+", False, None,   "K"),
    "sfcWind": ("+", True,  1e-6,   "m s-1"),
    "rsds":    ("+", True,  1e-6,   "W m-2"),
}

def load_era5(vname, units):
    ds = xr.open_dataset(ERA5_DIR / f"era5_{vname}_1980_2020.nc")
    da = ds[vname]
    da.attrs["units"] = units
    return da.sortby("lat").sortby("lon")

def load_cmip(var, scenario, units):
    pattern = str(CMIP_DIR / f"{var}_day_{GCM}_{scenario}_{RUN}*_india.nc")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files: {pattern}")
    da = xr.open_mfdataset(files, combine="by_coords")[var]
    da = da.drop_vars([c for c in da.coords
                       if c not in {"time","lat","lon","latitude","longitude"}],
                      errors="ignore")
    da["time"] = da.indexes["time"].floor("D")
    da.attrs["units"] = units
    return da.sortby("lat").sortby("lon").sortby("time")

def jitter_log(da, lower, unit):
    da = sdba_proc.jitter(da, lower=f"{lower} {unit}", minimum=f"0 {unit}")
    return sdba_proc.to_additive_space(da, lower_bound=f"0 {unit}", trans="log")


for vname, (kind, use_log, jitter_low, unit) in tqdm(VAR_CFG.items(), desc="Variable"):

    # ── ERA5 reference ───────────────────────────────────────────────────────────
    era5_da = load_era5(vname, unit)
    era5_da = convert_calendar(era5_da, "noleap", align_on="date").sel(time=TRAIN)

    # ── CMIP6 historical ─────────────────────────────────────────────────────────
    if vname == "sfcWind":
        _uas = load_cmip("uas", "historical", "m s-1")
        _vas = load_cmip("vas", "historical", "m s-1")
        hist_da = np.hypot(_uas, _vas).rename("sfcWind")
        hist_da.attrs["units"] = "m s-1"
        del _uas, _vas
    else:
        hist_da = load_cmip(vname, "historical", unit)
    hist_da = hist_da.sel(time=TRAIN)

    # ── Align grids (ERA5 0.25° → CanESM5 ~2.8°) and time axes ─────────────────
    era5_rg = era5_da.interp(lat=hist_da.lat, lon=hist_da.lon, method="linear")
    del era5_da

    era5_dates = pd.DatetimeIndex(era5_rg.time.values).normalize()
    hist_dates = pd.DatetimeIndex([pd.Timestamp(str(t)[:10]) for t in hist_da.time.values])
    common = era5_dates.intersection(hist_dates)
    print(f"  {vname}: {len(common)} common training days")

    era5_rg = era5_rg.sel(time=[t for t in era5_rg.time.values
                                 if pd.Timestamp(str(t)[:10]) in common]).load()
    hist_da = hist_da.sel(time=[t for t in hist_da.time.values
                                 if pd.Timestamp(str(t)[:10]) in common]).load()
    era5_rg = era5_rg.assign_coords(time=hist_da.time.values)

    # ── Train QDM once (historical vs ERA5) ─────────────────────────────────────
    if use_log:
        ref_v  = jitter_log(era5_rg, jitter_low, unit)
        hist_v = jitter_log(hist_da, jitter_low, unit)
    else:
        ref_v, hist_v = era5_rg, hist_da

    QM = sdba.QuantileDeltaMapping.train(
        ref_v, hist_v,
        nquantiles=50,
        kind=kind,
        group="time.month",
    )
    del ref_v, hist_v, era5_rg, hist_da

    # ── Apply to each SSP ────────────────────────────────────────────────────────
    for ssp in tqdm(SSPS, desc=f"  {vname} SSP", leave=False):
        out_path = OUTPUT_DIR / f"{vname}_{GCM}_{ssp}_bc.nc"
        if out_path.exists():
            print(f"  {ssp}/{vname}: exists — skipping")
            continue

        if vname == "sfcWind":
            _uas = load_cmip("uas", ssp, "m s-1")
            _vas = load_cmip("vas", ssp, "m s-1")
            fut_v = np.hypot(_uas, _vas).rename("sfcWind")
            fut_v.attrs["units"] = "m s-1"
            del _uas, _vas
        else:
            fut_v = load_cmip(vname, ssp, unit)
        fut_v = fut_v.load()

        if use_log:
            fut_v = jitter_log(fut_v, jitter_low, unit)

        bc_v = QM.adjust(fut_v, interp="linear", extrapolation="constant")
        del fut_v

        if use_log:
            bc_v = sdba_proc.from_additive_space(bc_v)
            bc_v.values[bc_v.values < 1e-5] = 0.0

        ds_out = bc_v.to_dataset(name=vname)
        ds_out[vname].attrs["units"] = unit
        ds_out.attrs = {
            "description": f"QDM bias-corrected {vname} — {GCM} {RUN} {ssp}",
            "method": "Quantile Delta Mapping (Cannon et al. 2015) via xclim.sdba",
            "reference": "ERA5 daily 1980–2010",
            "gcm": GCM, "run": RUN, "ssp": ssp,
        }
        ds_out.to_netcdf(out_path)
        del bc_v, ds_out
        print(f"  → {out_path.name}")

    del QM

print("\nBias correction complete.")

Variable:   0%|          | 0/4 [00:00<?, ?it/s]

## 5. Extract Global Warming Level (GWL) windows

In [ ]:
wl_dir = Path("../aux_data/mathause-cmip_warming_levels-f47853e/warming_levels/cmip6")
wl_files = (sorted(glob.glob(str(wl_dir / "csv" / "*1850_1900*.csv"))) or
            sorted(glob.glob(str(wl_dir / "**" / "*1850_1900*.csv"), recursive=True)))
wl_files.sort(key=lambda f: "all" not in f)

if not wl_files:
    print("Warming levels CSV not found — skipping")
else:
    wl = pd.read_csv(wl_files[0], comment="#", skipinitialspace=True)
    wl.columns = wl.columns.str.strip()
    print(f"Loaded: {wl_files[0]}\nColumns: {list(wl.columns)}")

    gwl_dir = OUTPUT_DIR / "gwl"
    gwl_dir.mkdir(exist_ok=True)

    rows = []
    for ssp in tqdm(SSPS, desc="SSP"):
        for gwl in [1.5, 2.0, 3.0, 4.0]:
            mask = ((wl["model"].str.strip() == GCM) &
                    (wl["exp"].str.strip() == ssp) &
                    (wl["warming_level"].astype(float) == gwl))
            if "ensemble" in wl.columns:
                mask &= wl["ensemble"].str.strip() == RUN
            row = wl[mask]
            if row.empty:
                row = wl[(wl["model"].str.strip() == GCM) &
                         (wl["exp"].str.strip() == ssp) &
                         (wl["warming_level"].astype(float) == gwl)]
            if row.empty:
                continue
            start, end = int(row.iloc[0]["start_year"]), int(row.iloc[0]["end_year"])
            rows.append({"ssp": ssp, "gwl": gwl, "start": start, "end": end})

            for vname in VAR_CFG:
                bc_file = OUTPUT_DIR / f"{vname}_{GCM}_{ssp}_bc.nc"
                out_gwl = gwl_dir / f"{vname}_{GCM}_{ssp}_GWL{str(gwl).replace('.','')}.nc"
                if not bc_file.exists() or out_gwl.exists():
                    continue
                ds = xr.open_dataset(bc_file)
                ds.sel(time=slice(str(start), str(end))).to_netcdf(out_gwl)

    pd.DataFrame(rows).to_csv(OUTPUT_DIR / "gwl_windows.csv", index=False)
    print(pd.DataFrame(rows).to_string(index=False))

## 6. Quick validation — domain-mean time series

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

PLOT_VARS = {
    "tas":     ("tas (K)",       "K"),
    "tasmax":  ("tasmax (K)",    "K"),
    "sfcWind": ("sfcWind (m/s)", "m s-1"),
    "rsds":    ("rsds (W/m²)",   "W m-2"),
}
COLORS = {"ssp245": "steelblue", "ssp585": "firebrick"}

for ax, (vname, (label, _)) in zip(axes, PLOT_VARS.items()):
    # ERA5 reference (annual mean, domain average)
    era5_da = xr.open_dataset(ERA5_DIR / f"era5_{vname}_1980_2020.nc")[vname]
    era5_ann = era5_da.mean(["lat", "lon"]).resample(time="YE").mean()
    ax.plot(era5_ann.time.dt.year, era5_ann.values, "k-", lw=1.5, label="ERA5", zorder=3)

    for ssp in SSPS:
        bc_file = OUTPUT_DIR / f"{vname}_{GCM}_{ssp}_bc.nc"
        if not bc_file.exists():
            continue
        bc = xr.open_dataset(bc_file)[vname]
        bc_ann = bc.mean(["lat", "lon"]).resample(time="YE").mean()
        ax.plot(bc_ann.time.dt.year, bc_ann.values,
                color=COLORS[ssp], lw=1, alpha=0.8, label=ssp)

    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.set_xlabel("Year")

plt.suptitle(f"Domain-mean annual series — {GCM} {RUN}", fontsize=12)
plt.tight_layout()
plt.show()